# CSV to SQLite Redesign

이 노트북은 기존 `final_dataset.csv`를 그대로 유지하면서, 후속 실험과 `5~9초` monitoring 구간 재구성을 쉽게 하기 위해 `SQLite` 기반 구조를 추가로 만드는 실험용 파이프라인입니다.

목표:
- 기존 CSV 경로 유지
- 대용량 CSV를 chunk 단위로 SQLite에 적재
- `video_id`, `time_sec`, `label` 기준 인덱스 생성
- `5~9초` monitoring 구간용 view / 집계 테이블 생성
- 이후 TCN/GRU 학습용 샘플 생성에 바로 연결할 수 있는 기반 확보


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/졸업 과제/dataset/processing/final_dataset.csv'
sqlite_path = '/content/drive/MyDrive/졸업 과제/dataset/processing/final_dataset.sqlite'

print(file_path)
print(sqlite_path)


In [ ]:
import os
import sqlite3
from pathlib import Path

import pandas as pd

CHUNK_SIZE = 100_000
TABLE_NAME = 'fall_frames'
MONITOR_START_SEC = 5.0
MONITOR_END_SEC = 9.0
TARGET_FPS = 15
CLIP_DURATION_SEC = 10.0
LABEL_POLICY = 'segment_max_on_5_to_9_sec'

Path(sqlite_path).parent.mkdir(parents=True, exist_ok=True)


## 1-1. 메타데이터 테이블 생성

DB 자체에 데이터셋/설계 메타정보를 저장합니다. 이후 학습 노트북은 이 테이블을 읽어 현재 DB가 어떤 규칙으로 만들어졌는지 확인할 수 있습니다.


In [ ]:
conn.execute('''
CREATE TABLE IF NOT EXISTS dataset_metadata (
    meta_key TEXT PRIMARY KEY,
    meta_value TEXT NOT NULL
);
''')

metadata_rows = {
    'source_csv_path': file_path,
    'sqlite_path': sqlite_path,
    'base_table': TABLE_NAME,
    'target_fps': str(TARGET_FPS),
    'clip_duration_sec': str(CLIP_DURATION_SEC),
    'monitor_start_sec': str(MONITOR_START_SEC),
    'monitor_end_sec': str(MONITOR_END_SEC),
    'target_monitor_steps': '60',
    'label_policy': LABEL_POLICY,
    'feature_engineering_notes': 'kp0~kp16 xyz-score style columns + HSSC_y HSSC_x RWHC VHSSC',
    'created_for': 'STM32N6 fall detection TFLite flow',
}

conn.executemany(
    'INSERT OR REPLACE INTO dataset_metadata(meta_key, meta_value) VALUES (?, ?)',
    list(metadata_rows.items())
)
conn.commit()

display(pd.read_sql_query('SELECT * FROM dataset_metadata ORDER BY meta_key;', conn))


## 1. CSV를 SQLite로 적재

대용량 CSV를 메모리에 한 번에 올리지 않고 chunk 단위로 넣습니다.


## 1-2. 컬럼 스키마 메타데이터 저장

원본 CSV의 컬럼 목록도 별도 테이블로 보관합니다.


In [ ]:
header_df = pd.read_csv(file_path, nrows=0)
schema_df = pd.DataFrame({
    'column_index': list(range(len(header_df.columns))),
    'column_name': list(header_df.columns),
})

schema_df.to_sql('dataset_columns', conn, if_exists='replace', index=False)
conn.execute('CREATE INDEX IF NOT EXISTS idx_dataset_columns_name ON dataset_columns(column_name);')
conn.commit()

display(pd.read_sql_query('SELECT * FROM dataset_columns LIMIT 10;', conn))


In [ ]:
if os.path.exists(sqlite_path):
    os.remove(sqlite_path)

conn = sqlite3.connect(sqlite_path)

total_rows = 0
for chunk_idx, chunk in enumerate(pd.read_csv(file_path, chunksize=CHUNK_SIZE)):
    chunk.to_sql(TABLE_NAME, conn, if_exists='append', index=False)
    total_rows += len(chunk)
    print(f'chunk={chunk_idx:03d} rows_loaded={total_rows}')

conn.commit()
print(f'load complete: {total_rows:,} rows')


## 2. 인덱스 생성

이후 필요한 쿼리는 거의 `video_id`, `time_sec`, `label` 기준이라 여기에 맞춰 인덱스를 만듭니다.


In [ ]:
index_sql = [
    f'CREATE INDEX IF NOT EXISTS idx_{TABLE_NAME}_video_id ON {TABLE_NAME}(video_id);',
    f'CREATE INDEX IF NOT EXISTS idx_{TABLE_NAME}_video_time ON {TABLE_NAME}(video_id, time_sec);',
    f'CREATE INDEX IF NOT EXISTS idx_{TABLE_NAME}_label ON {TABLE_NAME}(label);',
    f'CREATE INDEX IF NOT EXISTS idx_{TABLE_NAME}_time ON {TABLE_NAME}(time_sec);',
]

for sql in index_sql:
    conn.execute(sql)

conn.commit()
print('index creation complete')


## 3. 기본 통계 확인

SQLite로 옮긴 뒤 바로 검증합니다.


In [ ]:
queries = {
    'total_rows': f'SELECT COUNT(*) AS total_rows FROM {TABLE_NAME};',
    'unique_videos': f'SELECT COUNT(DISTINCT video_id) AS unique_videos FROM {TABLE_NAME};',
    'label_distribution': f'SELECT label, COUNT(*) AS frame_count FROM {TABLE_NAME} GROUP BY label ORDER BY label;',
}

for name, sql in queries.items():
    print(f'[{name}]')
    display(pd.read_sql_query(sql, conn))


## 4. `5~9초` monitoring 구간 view 생성

현재 모델링 전략에 맞춰 `5초 이상 9초 미만` 구간을 공통 monitoring 구간으로 잡습니다.


In [ ]:
conn.execute('DROP VIEW IF EXISTS monitoring_frames;')
conn.execute(f'''
CREATE VIEW monitoring_frames AS
SELECT *
FROM {TABLE_NAME}
WHERE time_sec >= {MONITOR_START_SEC} AND time_sec < {MONITOR_END_SEC};
''')
conn.commit()

display(pd.read_sql_query('SELECT COUNT(*) AS monitoring_rows FROM monitoring_frames;', conn))


## 5. 클립 단위 집계 테이블 생성

`video_id`별로 monitoring 구간의 시작/끝 시간, 프레임 수, positive 존재 여부를 집계합니다.
이 테이블은 이후 clip classification 학습용 메타테이블로 사용합니다.


In [ ]:
conn.execute('DROP TABLE IF EXISTS monitoring_segments;')
conn.execute('''
CREATE TABLE monitoring_segments AS
SELECT
    video_id,
    MIN(time_sec) AS segment_start_sec,
    MAX(time_sec) AS segment_end_sec,
    COUNT(*) AS segment_frames,
    MAX(label) AS segment_label,
    SUM(CASE WHEN label = 1 THEN 1 ELSE 0 END) AS positive_frame_count,
    AVG(CASE WHEN label = 1 THEN 1.0 ELSE 0.0 END) AS positive_frame_ratio
FROM monitoring_frames
GROUP BY video_id;
''')
conn.execute('CREATE INDEX IF NOT EXISTS idx_monitoring_segments_video_id ON monitoring_segments(video_id);')
conn.execute('CREATE INDEX IF NOT EXISTS idx_monitoring_segments_label ON monitoring_segments(segment_label);')
conn.commit()

display(pd.read_sql_query('SELECT * FROM monitoring_segments LIMIT 5;', conn))
display(pd.read_sql_query('SELECT segment_label, COUNT(*) AS clips FROM monitoring_segments GROUP BY segment_label ORDER BY segment_label;', conn))


## 6. 학습용 샘플 로딩 예시

아래 쿼리는 특정 `video_id` 하나의 monitoring 구간을 불러오는 예시입니다.


In [ ]:
sample_video = pd.read_sql_query('SELECT video_id FROM monitoring_segments LIMIT 1;', conn).iloc[0, 0]
sample_sql = f'''
SELECT *
FROM monitoring_frames
WHERE video_id = ?
ORDER BY time_sec, frame;
'''
sample_df = pd.read_sql_query(sample_sql, conn, params=[sample_video])
print(sample_video, sample_df.shape)
display(sample_df.head())


## 7. SQLite 재설계의 장단점

장점:
- `video_id` 기준 집계와 분할이 쉬움
- `5~9초` 같은 공통 monitoring 구간을 view로 고정 가능
- 향후 onset metadata가 복구되면 별도 테이블로 붙이기 쉬움
- clip 단위 메타테이블을 만들기 편함
- 데이터셋 규칙 자체를 `dataset_metadata`로 함께 저장할 수 있음

주의점:
- 학습 자체는 여전히 최종적으로 `numpy` 또는 `tf.data` 텐서로 변환해야 함
- SQLite는 분석/조회/정합성 관리에는 좋지만, 딥러닝 학습 중 직접 random access source로 쓰기에는 느릴 수 있음
- 따라서 추천 구조는 `CSV -> SQLite 정리 -> clip table 확정 -> 학습용 numpy export` 입니다.


In [ ]:
conn.close()
print('done')
